1. Typage des colonnes et chargement des donnees

In [ ]:
import pandas as pd

dtypes = {
    "iu_ac": "string",
    "libelle": "string",
    "iu_nd_amont": "string",
    "libelle_nd_amont": "string",
    "iu_nd_aval": "string",
    "libelle_nd_aval": "string",
    "etat_trafic": "category",
}

# Chargement avec conversion des dates en UTC puis conversion en heure de Paris
df = pd.read_csv(
    "../data/raw/comptages.csv",
    sep=";",
    dtype=dtypes,
    parse_dates=["t_1h"],
)

# Gestion du fuseau horaire : on passe du fuseau UTC vers le fuseau Europe/Paris
df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True).dt.tz_convert("Europe/Paris")

# Filtrage sur tes 3 axes
axes = ["Av_Foch", "Bd_Grenelle", "Bd_Barbes"]
df = df[df["libelle"].isin(axes)].copy()

print("Format des dates et types validés :")
print(df.dtypes[["t_1h", "iu_ac", "etat_trafic"]])

Format des dates et types validés :
t_1h           datetime64[us, Europe/Paris]
iu_ac                                string
etat_trafic                        category
dtype: object


2. Test et suppression des doublons

In [ ]:
# 1. Nettoyage préliminaire des chaînes pour éviter les doublons cachés par espaces
df["iu_ac"] = df["iu_ac"].str.strip()

# 2. Test des doublons stricts sur la clé d'unicité naturelle
cle_unicite = ["iu_ac", "t_1h"]
nb_doublons = df.duplicated(subset=cle_unicite).sum()
print(f"Nombre de doublons stricts détectés sur (iu_ac, t_1h) : {nb_doublons}")

# 3. Test des doublons cachés (on fait un arrondi temporel à l'heure pile)
df["t_1h_ronde"] = df["t_1h"].dt.floor("h")
nb_doublons_caches = df.duplicated(subset=["iu_ac", "t_1h_ronde"]).sum()
print(f"Nombre de doublons cachés (arrondi horaire) : {nb_doublons_caches}")

# 4. Suppression si nécessaire et nettoyage de la colonne temporaire
if nb_doublons > 0 or nb_doublons_caches > 0:
    df = df.drop_duplicates(subset=["iu_ac", "t_1h_ronde"], keep="first")
    print("Doublons (stricts ou cachés) supprimés avec succès.")
else:
    print("Aucun doublon strict ni caché détecté.")

df = df.drop(columns=["t_1h_ronde"])

Nombre de doublons stricts détectés sur (iu_ac, t_1h) : 0
Nombre de doublons cachés (arrondi horaire) : 0
Aucun doublon strict ni caché détecté.
